# 📈 Notebook 2 — Exploratory Data Analysis

We explore the merged dataset visually and statistically to understand how load shedding
affected revenue across the 3-month period.

**Questions we answer:**
1. What did daily revenue look like over time?
2. Which months were worst affected?
3. How did each load shedding stage affect revenue?
4. Which days of the week were most impacted?
5. How much total revenue was lost?

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/merged.csv', parse_dates=['date'])
print('Data loaded:', df.shape)
df.head()

## 1 — Daily Revenue Over Time

We plot daily revenue as a line chart and colour-code each point by load shedding stage. This immediately shows the visual correlation between heavy load shedding and revenue dips.

In [ ]:
fig = px.scatter(
    df[df['revenue'] > 0],
    x='date', y='revenue',
    color='stage_category',
    title='Daily Revenue with Load Shedding Stage (Jan–Mar 2024)',
    labels={'revenue': 'Revenue (R)', 'date': 'Date', 'stage_category': 'LS Stage'},
    color_discrete_map={
        'None':        '#2ecc71',
        'Low (1-2)':   '#f39c12',
        'Medium (3-4)':'#e67e22',
        'High (5-6)':  '#e74c3c'
    },
    template='plotly_dark',
    size_max=8
)
fig.add_hline(y=df['baseline_revenue'].iloc[0], line_dash='dash',
              line_color='white', annotation_text='Baseline Revenue')
fig.show()

## 2 — Monthly Revenue Summary

January and February 2024 were the peak load shedding months in South Africa — we expect to see lower revenue in those months compared to March when load shedding began easing.

In [ ]:
monthly = df[df['revenue'] > 0].groupby('month').agg(
    total_revenue=('revenue','sum'),
    avg_daily_revenue=('revenue','mean'),
    total_loss=('revenue_loss','sum'),
    ls_days=('is_loadshedding','sum')
).reset_index()

# Sort by month order
month_order = ['January','February','March']
monthly['month'] = pd.Categorical(monthly['month'], categories=month_order, ordered=True)
monthly = monthly.sort_values('month')

print('Monthly Summary:')
print(monthly.to_string(index=False))

fig = px.bar(monthly, x='month', y=['total_revenue','total_loss'],
             title='Monthly Revenue vs Revenue Lost to Load Shedding',
             labels={'value':'Amount (R)', 'variable':''},
             barmode='group',
             color_discrete_sequence=['#3498db','#e74c3c'],
             template='plotly_dark')
fig.show()

## 3 — Average Revenue by Load Shedding Stage

This is the core finding of the analysis. We group all trading days by their load shedding stage and calculate the average revenue for each. The drop from Stage 0 to Stage 6 quantifies the exact cost of each stage increment.

In [ ]:
stage_rev = df[
    (df['revenue'] > 0) & (df['is_public_holiday'] == False)
].groupby('stage').agg(
    avg_revenue=('revenue','mean'),
    days=('revenue','count')
).reset_index()

fig = px.bar(stage_rev, x='stage', y='avg_revenue',
             color='avg_revenue',
             title='Average Daily Revenue by Load Shedding Stage',
             labels={'avg_revenue': 'Avg Revenue (R)', 'stage': 'Load Shedding Stage'},
             color_continuous_scale='RdYlGn',
             template='plotly_dark',
             text='days')
fig.update_traces(texttemplate='%{text} days', textposition='outside')
fig.show()

print('\nKey finding:')
stage0 = stage_rev[stage_rev['stage']==0]['avg_revenue'].values[0]
for _, row in stage_rev.iterrows():
    pct = ((row['avg_revenue'] - stage0) / stage0) * 100
    print(f'  Stage {int(row["stage"])}: R {row["avg_revenue"]:,.0f}/day  ({pct:+.1f}% vs baseline)')

## 4 — Revenue Loss Distribution

A box plot showing how revenue varied by stage category. The wider the box and the lower the median, the more unpredictable and damaging that stage was.

In [ ]:
trading = df[(df['revenue'] > 0) & (df['is_public_holiday'] == False)]
order = ['None','Low (1-2)','Medium (3-4)','High (5-6)']
fig = px.box(trading, x='stage_category', y='revenue',
             category_orders={'stage_category': order},
             title='Revenue Distribution by Load Shedding Category',
             labels={'revenue':'Revenue (R)', 'stage_category':'Stage Category'},
             color='stage_category',
             color_discrete_map={'None':'#2ecc71','Low (1-2)':'#f39c12',
                                  'Medium (3-4)':'#e67e22','High (5-6)':'#e74c3c'},
             template='plotly_dark')
fig.show()

## 5 — Hours Affected vs Revenue (Scatter)

Each point is one trading day. The trend line shows the relationship between hours of load shedding and revenue earned. A downward slope confirms the negative correlation.

In [ ]:
import plotly.express as px
trading = df[(df['revenue'] > 0) & (df['is_public_holiday'] == False)]
fig = px.scatter(trading, x='hours_affected', y='revenue',
                 trendline='ols',
                 color='month',
                 title='Hours of Load Shedding vs Daily Revenue',
                 labels={'hours_affected':'Hours Affected','revenue':'Revenue (R)'},
                 template='plotly_dark')
fig.show()

corr = trading['hours_affected'].corr(trading['revenue'])
print(f'Pearson correlation (hours vs revenue): {corr:.3f}')
print('Interpretation: Values close to -1 = strong negative relationship')

## 6 — Total Revenue Loss Summary

Final summary of the estimated financial impact across the 3-month period.

In [ ]:
baseline = df['baseline_revenue'].iloc[0]
total_loss  = df['revenue_loss'].sum()
ls_days     = (df['is_loadshedding'] & (df['revenue'] > 0) & (df['is_public_holiday']==False)).sum()
avg_loss    = total_loss / ls_days if ls_days > 0 else 0

print('='*55)
print('REVENUE IMPACT SUMMARY — JAN to MAR 2024')
print('='*55)
print(f'Baseline daily revenue       : R {baseline:,.2f}')
print(f'Total trading days           : {(df["revenue"]>0).sum()}')
print(f'Load shedding trading days   : {int(ls_days)}')
print(f'Total revenue lost           : R {total_loss:,.2f}')
print(f'Avg loss per affected day    : R {avg_loss:,.2f}')
print(f'Avg loss as % of baseline    : {(avg_loss/baseline*100):.1f}%')
print('='*55)